In [1]:
!pip install -q tabpfn==9.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 49.8 MB/s eta 0:00:00


In [2]:
import torch
import pandas as pd
import numpy as np

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [4]:
df = pd.read_csv(
    "/content/training_samples_union_with_splits.csv"
)

FEATURES = [
    "aspect",
    "curvature",
    "distdrainage",
    "distriver",
    "distroads",
    "elevation",
    "ndvi",
    "rainfall_frequency",
    "slope",
    "twi",
]

X = df[FEATURES]
y = df["class"].astype(int)

print(df.shape)
print(y.value_counts())

(20000, 20)
class
1    10000
0    10000
Name: count, dtype: int64


In [6]:
import os
from getpass import getpass

os.environ["TABPFN_TOKEN"] = getpass(
    "Paste your TabPFN API key here: "
)

print("Token set.")

Paste your TabPFN API key here: ··········
Token set.


In [7]:
import pandas as pd
import numpy as np
import torch

from tabpfn import TabPFNClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)

FEATURES = [
    "aspect",
    "curvature",
    "distdrainage",
    "distriver",
    "distroads",
    "elevation",
    "ndvi",
    "rainfall_frequency",
    "slope",
    "twi",
]

# ------------------------------------------------------------
# Random split
# ------------------------------------------------------------

train_mask = df["random_split"] == "train"
test_mask = df["random_split"] == "test"

X_train = df.loc[train_mask, FEATURES]
y_train = df.loc[train_mask, "class"].astype(int)

X_test = df.loc[test_mask, FEATURES]
y_test = df.loc[test_mask, "class"].astype(int)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# ------------------------------------------------------------
# TabPFN
# ------------------------------------------------------------

model = TabPFNClassifier(
    device="cuda",
    random_state=42,
    show_progress_bar=True,
)

print("\nFitting TabPFN...")

model.fit(
    X_train,
    y_train,
)

print("\nPredicting...")

probability = model.predict_proba(
    X_test
)[:, 1]

prediction = (
    probability >= 0.5
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

metrics = {
    "roc_auc": roc_auc_score(
        y_test,
        probability
    ),

    "pr_auc": average_precision_score(
        y_test,
        probability
    ),

    "accuracy": accuracy_score(
        y_test,
        prediction
    ),

    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        prediction
    ),

    "precision": precision_score(
        y_test,
        prediction
    ),

    "recall": recall_score(
        y_test,
        prediction
    ),

    "f1": f1_score(
        y_test,
        prediction
    ),

    "brier": brier_score_loss(
        y_test,
        probability
    ),
}

print("\n" + "=" * 60)
print("TABPFN RANDOM HOLD-OUT")
print("=" * 60)

for metric, value in metrics.items():
    print(
        f"{metric:<20}"
        f"{value:.4f}"
    )

Train: (15400, 10)
Test: (4600, 10)

Fitting TabPFN...


tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]


Predicting...


TabPFN inference:   0%|          | 0/8 [00:00<?, ?estimator/s]


TABPFN RANDOM HOLD-OUT
roc_auc             0.9523
pr_auc              0.9472
accuracy            0.8957
balanced_accuracy   0.8957
precision           0.8856
recall              0.9087
f1                  0.8970
brier               0.0815


In [8]:
import pandas as pd
import numpy as np
import torch

from tabpfn import TabPFNClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(y_true, probability):

    prediction = (
        probability >= 0.5
    ).astype(int)

    return {
        "roc_auc": roc_auc_score(
            y_true,
            probability
        ),

        "pr_auc": average_precision_score(
            y_true,
            probability
        ),

        "accuracy": accuracy_score(
            y_true,
            prediction
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            prediction
        ),

        "precision": precision_score(
            y_true,
            prediction
        ),

        "recall": recall_score(
            y_true,
            prediction
        ),

        "f1": f1_score(
            y_true,
            prediction
        ),

        "brier": brier_score_loss(
            y_true,
            probability
        ),
    }


# ============================================================
# 5-FOLD SPATIAL CV
# ============================================================

print("=" * 70)
print("TABPFN — 5-FOLD SPATIAL BLOCK CV")
print("=" * 70)

spatial_results = []


for fold in range(1, 6):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}")
    print("=" * 70)

    train_mask = (
        df["spatial_fold"] != fold
    )

    validation_mask = (
        df["spatial_fold"] == fold
    )


    X_train = df.loc[
        train_mask,
        FEATURES
    ]

    y_train = df.loc[
        train_mask,
        "class"
    ].astype(int)


    X_validation = df.loc[
        validation_mask,
        FEATURES
    ]

    y_validation = df.loc[
        validation_mask,
        "class"
    ].astype(int)


    print(
        "Train:",
        X_train.shape
    )

    print(
        "Validation:",
        X_validation.shape
    )


    # --------------------------------------------------------
    # New TabPFN instance for each fold
    # --------------------------------------------------------

    model = TabPFNClassifier(
        device="cuda",
        random_state=42,
        show_progress_bar=True,
    )


    print("\nFitting...")

    model.fit(
        X_train,
        y_train
    )


    print("Predicting...")

    probability = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )


    metrics = calculate_metrics(
        y_validation,
        probability
    )


    metrics["fold"] = fold

    metrics["n_validation"] = len(
        y_validation
    )


    spatial_results.append(
        metrics
    )


    print("\nResults:")

    for metric in [
        "roc_auc",
        "pr_auc",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "brier",
    ]:

        print(
            f"{metric:<20}"
            f"{metrics[metric]:.4f}"
        )


    # Free GPU memory before next fold
    del model

    torch.cuda.empty_cache()


# ============================================================
# SUMMARY
# ============================================================

spatial_df = pd.DataFrame(
    spatial_results
)


print("\n" + "=" * 70)
print("TABPFN SPATIAL CV MEAN ± SD")
print("=" * 70)


summary_rows = []


for metric in [
    "roc_auc",
    "pr_auc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "brier",
]:

    mean = spatial_df[
        metric
    ].mean()

    sd = spatial_df[
        metric
    ].std(ddof=1)


    print(
        f"{metric:<20}"
        f"{mean:.4f} ± {sd:.4f}"
    )


    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "sd": sd
    })


# ============================================================
# SAVE RESULTS
# ============================================================

spatial_df.to_csv(
    "/content/tabpfn_spatial_fold_metrics.csv",
    index=False
)


pd.DataFrame(
    summary_rows
).to_csv(
    "/content/tabpfn_spatial_summary.csv",
    index=False
)


print(
    "\nSaved:"
    "\n/content/tabpfn_spatial_fold_metrics.csv"
    "\n/content/tabpfn_spatial_summary.csv"
)

TABPFN — 5-FOLD SPATIAL BLOCK CV

FOLD 1
Train: (16072, 10)
Validation: (3928, 10)

Fitting...
Predicting...


TabPFN inference:   0%|          | 0/8 [00:00<?, ?estimator/s]


Results:
roc_auc             0.9144
pr_auc              0.9034
accuracy            0.8383
balanced_accuracy   0.8384
precision           0.8566
recall              0.8135
f1                  0.8345
brier               0.1189

FOLD 2
Train: (16060, 10)
Validation: (3940, 10)

Fitting...
Predicting...


TabPFN inference:   0%|          | 0/8 [00:00<?, ?estimator/s]


Results:
roc_auc             0.9216
pr_auc              0.9069
accuracy            0.8282
balanced_accuracy   0.8273
precision           0.8662
recall              0.7702
f1                  0.8154
brier               0.1187

FOLD 3
Train: (15951, 10)
Validation: (4049, 10)

Fitting...
Predicting...


...


Results:
roc_auc             0.9232
pr_auc              0.9225
accuracy            0.8390
balanced_accuracy   0.8393
precision           0.8829
recall              0.7841
f1                  0.8306
brier               0.1148

FOLD 4
Train: (15918, 10)
Validation: (4082, 10)

Fitting...
Predicting...


TabPFN inference:   0%|          | 0/8 [00:00<?, ?estimator/s]


Results:
roc_auc             0.9117
pr_auc              0.9092
accuracy            0.8146
balanced_accuracy   0.8141
precision           0.9013
recall              0.7049
f1                  0.7911
brier               0.1351

FOLD 5
Train: (15999, 10)
Validation: (4001, 10)

Fitting...
Predicting...


TabPFN inference:   0%|          | 0/8 [00:00<?, ?estimator/s]


Results:
roc_auc             0.8801
pr_auc              0.8625
accuracy            0.7846
balanced_accuracy   0.7853
precision           0.8374
recall              0.7114
f1                  0.7693
brier               0.1522

TABPFN SPATIAL CV MEAN ± SD
roc_auc             0.9102 ± 0.0175
pr_auc              0.9009 ± 0.0226
accuracy            0.8209 ± 0.0226
balanced_accuracy   0.8209 ± 0.0224
precision           0.8689 ± 0.0245
recall              0.7568 ± 0.0472
f1                  0.8082 ± 0.0276
brier               0.1279 ± 0.0157

Saved:
/content/tabpfn_spatial_fold_metrics.csv
/content/tabpfn_spatial_summary.csv


In [9]:
tabpfn_random = pd.DataFrame([{
    "roc_auc": 0.9523,
    "pr_auc": 0.9472,
    "accuracy": 0.8957,
    "balanced_accuracy": 0.8957,
    "precision": 0.8856,
    "recall": 0.9087,
    "f1": 0.8970,
    "brier": 0.0815
}])

tabpfn_random.to_csv(
    "/content/tabpfn_random_metrics.csv",
    index=False
)

In [10]:
from google.colab import files

files.download(
    "/content/tabpfn_random_metrics.csv"
)

files.download(
    "/content/tabpfn_spatial_fold_metrics.csv"
)

files.download(
    "/content/tabpfn_spatial_summary.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>